# Per-perturbation neural bridge: leave-one-batch-out evaluation

Set `n_validation_batches` below to choose how many distinct batches to evaluate, one at a time. Train a fresh per-perturbation neural bridge and the fitted-linear model for each selected batch, using **every other batch** for training. This gives `n_validation_batches` independent fits per model; batches held out in other runs remain available for training.

Only control inputs are standardised, using that run's training-control statistics. Targets and predictions stay in the dataset's existing expression units. The neural model's inner cross-fitting remains enabled.

Each held-out batch gets training and validation MSE curves for the neural bridge. A separate plot shows **neural MSE − fitted-linear MSE** against training-batch count, with mean and median differences at each count. For this comparison, **below zero means neural is better; above zero means neural is worse.**

In [1]:
import sys
from datetime import datetime
from urllib.parse import quote

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.ticker import MaxNLocator
from omegaconf import OmegaConf

from genetic_perturbation_playground.data.data_loader import DATA_DIR, load_dataset
from genetic_perturbation_playground.model.batch_transfer import (
    BatchTransferModel,
    _bridge_data,
)
from genetic_perturbation_playground.utils.metrics import mse
from genetic_perturbation_playground.utils.plotting import STYLE

sys.path.insert(0, str(DATA_DIR.parent / "scripts"))
from train_batch_transfer_expression import _make_bridge, _split_controls, cross_validate

n_validation_batches = 1  # n separate runs, each holding out one batch

cfg = OmegaConf.merge(
    OmegaConf.load(DATA_DIR.parent / "configs" / "train_batch_transfer_expression.yaml"),
    {
        "name": "perx_leave_one_batch_out",
        "n_validation_batches": n_validation_batches,
        "seed": 0,
        "top_k": 100,
        "bridges": ["neural"],
        "bridge_reg": 0.0001,
        "neural": {
            "per_perturbation": True,
            "weight_decay": 0.0001,
            "n_features_w": 8,
            "width_w": 8,
            "depth": 2,
            "lr": 0.00075,
            "n_steps": 400,
            "cross_fit_folds": 5,
            "device": "cuda" if torch.cuda.is_available() else "cpu",
        },
    },
)
output_dir = DATA_DIR.parent / "outputs" / f"{cfg.name}_{datetime.now():%Y%m%d_%H%M%S}"

## Data and control split

The script selects the validation batches and splits each batch's controls into two halves once. Models use the seen half as inputs; scoring uses the other half as the control baseline. All perturbed cells in the remaining batches enter training.

In [2]:
adata = load_dataset()
n_batches = adata.obs["batch"].nunique()
print(
    f"{cfg.n_validation_batches} separate runs: one held-out batch and {n_batches - 1} training batches each"
)

Loaded /nfs/ghome/live/adhir/genetic_playground/data/replogle_k562_preprocessed.h5ad
  310,385 cells  ×  8,563 genes  |  2,058 perturbations
1 separate runs: one held-out batch and 47 training batches each


## Train and record held-out learning curves

Step 0 scores the randomly initialized embedding with ridge coefficients fitted on the training batches. The curve then scores the same bridge after every optimizer update. Each run fits the linear reference once on the same training set.

Training MSE is the optimizer's cell-count-weighted objective on fresh cross-fitting folds of the training batches. Validation MSE averages errors equally over known perturbations in the held-out batch and matches the final score table. The held-out batch contributes only control inputs to prediction. The final checkpoint is used regardless of either curve.

In [ ]:
def fit_with_learning_curve(train, cfg, test, seen):
    """Score the neural bridge at initialization and after each optimizer update.

    Record mean per-perturbation validation MSE and the fitted-linear reference on the same
    known perturbations in this held-out batch. Only training responses enter fitting; held-out
    scores are diagnostics and never select a checkpoint. Use the normal prediction method
    so checkpoint and final losses are both measured in the original target units.
    """
    held_out = np.unique(test.obs["batch"]).item()
    linear = BatchTransferModel(standardise_w=cfg.standardise_w).fit(train)
    bridge = _make_bridge("neural", cfg)
    inputs, _ = _split_controls(test, seen)
    pairs = _bridge_data(inputs)
    labels = pairs.perturbations[pairs.pert]
    known = np.isin(labels, linear.perturbations_)
    labels, truth = labels[known], pairs.y[known]
    linear_mse = mse(linear.predict(labels, inputs), truth).mean()
    validation_mse, training_mse = {}, {}

    def record(step):
        validation_mse[step] = mse(bridge.predict(labels, inputs), truth).mean()

    steps = range(cfg.neural.n_steps + 1)
    bridge.fit(train, steps, record, on_training_loss=training_mse.__setitem__)
    for step in steps:
        curve_rows.append({
            "batch": held_out, "step": step, "training_mse": training_mse[step],
            "neural_mse": validation_mse[step], "linear_mse": linear_mse,
            "mse_difference": validation_mse[step] - linear_mse,
        })
    return {"transfer (fit b)": linear, "neural bridge": bridge}

In [ ]:
curve_rows = []
_, per_batch, scores = cross_validate(adata, cfg, fit_models=fit_with_learning_curve)
learning_curves = pd.DataFrame(curve_rows)
losses = scores.query("metric == 'mse'").pivot(
    index=["batch", "perturbation", "n_training_batches"], columns="model", values="value"
)
differences = (
    (losses["neural bridge"] - losses["transfer (fit b)"]).rename("mse_difference").reset_index()
)
output_dir.mkdir(parents=True, exist_ok=True)
scores.to_csv(output_dir / "scores.csv", index=False)
learning_curves.to_csv(output_dir / "learning_curves.csv", index=False)
differences.to_csv(output_dir / "mse_differences.csv", index=False)
print(f"Saved scores and loss differences to {output_dir}")

validation 1/1: holding out 13


neural bridge:  20%|██        | 80/400 [01:04<04:18,  1.24it/s, loss=0.0114]

## Neural training and validation MSE

Training MSE is shown above validation MSE. The panels share a step axis and use independent MSE scales. Dots mark the values before the first optimizer update. The last validation point equals the batch’s final neural MSE.


In [ ]:
def plot_learning_curves(table, title, path):
    fig, axes = plt.subplots(2, 1, figsize=(7, 5.2), sharex=True, layout="constrained")
    fig.suptitle(title)
    for ax, metric, label, color in [
        (axes[0], "training_mse", "Training (cross-fitted)", "C1"),
        (axes[1], "neural_mse", "Validation (held-out)", "C0"),
    ]:
        ax.plot(table["step"], table[metric], color=color, marker="o", markevery=[0])
        ax.set_title(label, loc="left")
        ax.set_ylabel("Mean squared error")
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
    axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[1].set_xlabel("Completed optimizer updates")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


curve_dir = output_dir / "plots" / "learning_curves"
curve_dir.mkdir(parents=True, exist_ok=True)
by_batch = learning_curves.groupby("batch", sort=False)
with plt.rc_context(STYLE):
    for held_out, table in by_batch:
        path = curve_dir / f"batch_{quote(str(held_out), safe='')}.png"
        plot_learning_curves(table.sort_values("step"), f"Held-out batch {held_out}", path)
    if by_batch.ngroups > 1:
        mean_curve = learning_curves.groupby("step", as_index=False)[["training_mse", "neural_mse"]].mean()
        plot_learning_curves(mean_curve, f"Mean across {by_batch.ngroups} held-out batches",
                             output_dir / "training_and_validation.png")

## Final loss difference by training-batch count

Each dot is one perturbation's **neural MSE − fitted-linear MSE** in this held-out batch. The x-axis counts training batches containing that perturbation, excluding the held-out batch. Lines show the **mean and median** across perturbations at each training-batch count. Held-out batches remain separate. Negative differences favour the neural bridge; positive differences favour the fitted-linear model.


In [ ]:
plot_dir = output_dir / "plots" / "by_training_batches"
plot_dir.mkdir(parents=True, exist_ok=True)
with plt.rc_context(STYLE):
    for held_out, table in differences.groupby("batch", sort=False):
        fig, ax = plt.subplots(figsize=(6.4, 3.8), layout="constrained")
        ax.scatter(
            table["n_training_batches"],
            table["mse_difference"],
            s=12,
            alpha=0.4,
            label="Perturbations",
        )
        summary = table.groupby("n_training_batches")["mse_difference"].agg(["mean", "median"])
        ax.plot(summary.index, summary["mean"], color="C1", marker="o", label="Mean")
        ax.plot(
            summary.index, summary["median"], color="C2", marker="s", linestyle="--", label="Median"
        )
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.set(
            title=f"Held-out batch: {held_out}",
            xlabel="Training batches containing the perturbation",
            ylabel="Neural MSE − fitted-linear MSE",
            xticks=sorted(table["n_training_batches"].unique()),
        )
        ax.legend(frameon=False)
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
        fig.savefig(
            plot_dir / f"batch_{quote(str(held_out), safe='')}.png", dpi=300, bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)

## Mean final gain or loss for each held-out batch

This table averages the individual MSE differences within each batch. Negative means the neural bridge has lower MSE than the fitted-linear model.

In [ ]:
differences.groupby("batch", sort=False)["mse_difference"].mean().to_frame().style.format("{:+.4g}")